# Data Engineering — Moduł 3: Apache Airflow — zaawansowane wzorce

W Module 2 poznaliśmy podstawowe klocki Airflow: DAG, Task, zależności, XCom-y. Ten moduł
dodaje wzorce, których używa się w PRAWDZIWYCH, produkcyjnych potokach: czekanie na
warunki (sensory), wybór ścieżki w zależności od danych (branching), dynamiczne
generowanie zadań (dynamic task mapping) oraz wywoływanie narzędzi zewnętrznych (dbt,
Spark) z poziomu DAG-a.

## Spis treści
1. [Sensory — czekanie na warunek](#sec1)
2. [Branching — wybór ścieżki w zależności od danych](#sec2)
3. [Dynamic Task Mapping](#sec3)
4. [Retries i trigger rules](#sec4)
5. [Wywoływanie narzędzi zewnętrznych z Airflow](#sec5)
6. [Podsumowanie i ćwiczenia](#sec6)


Uruchamiamy Airflow lokalnie -- w prawdziwym wdrożeniu `AIRFLOW_HOME` i baza metadanych (tu: SQLite, produkcyjnie zwykle PostgreSQL) są konfigurowane raz dla całego zespołu, a nie w każdym notebooku osobno. Robimy to tutaj, żeby każdy moduł kursu działał niezależnie od pozostałych.

In [ ]:
import os
import subprocess

# Kazdy modul dotyczacy Airflow uzywa wlasnego, izolowanego AIRFLOW_HOME (katalogu
# roboczego z metadanymi, DAG-ami i logami) -- tak, jak w prawdziwym wdrozeniu, gdzie
# AIRFLOW_HOME jest ustawiane raz dla calego srodowiska.
AIRFLOW_HOME = os.path.abspath("airflow_home")
AIRFLOW_BIN = "/tmp/airflow_venv/bin/airflow"
os.makedirs(f"{AIRFLOW_HOME}/dags", exist_ok=True)

env = os.environ.copy()
env["AIRFLOW_HOME"] = AIRFLOW_HOME
env["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
env["AIRFLOW__LOGGING__LOGGING_LEVEL"] = "WARNING"


def airflow(*args, check=True):
    """Uruchamia komende `airflow ...` w izolowanym AIRFLOW_HOME i wypisuje wynik."""
    wynik = subprocess.run([AIRFLOW_BIN, *args], env=env, capture_output=True, text=True)
    print(wynik.stdout)
    if wynik.returncode != 0:
        print(wynik.stderr[-3000:])
        if check:
            raise RuntimeError(f"airflow {' '.join(args)} zakonczyl sie kodem {wynik.returncode}")
    return wynik


# Inicjalizacja bazy metadanych (SQLite) -- potrzebna raz na AIRFLOW_HOME
airflow("db", "migrate")


<a id="sec1"></a>
## 1. Sensory — czekanie na warunek

**Sensor** to specjalny typ Operatora, który CYKLICZNIE sprawdza warunek ("czy plik już
się pojawił?", "czy inny DAG się zakończył?", "czy jest już określona godzina?") i
kończy się sukcesem dopiero, gdy warunek jest spełniony. `PythonSensor` sprawdza dowolną
funkcję Pythona zwracającą `True`/`False`; wbudowane sensory (np. do plików, do innych
DAG-ów) działają analogicznie, ale sprawdzają konkretny, gotowy warunek.

In [ ]:
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.sensors.python import PythonSensor
from datetime import datetime
import os


def plik_gotowy():
    # w prawdziwym potoku: sprawdzenie, czy zewnetrzny system juz zapisal plik
    gotowy = os.path.exists("dane_wejsciowe.csv")
    print(f"Sprawdzam plik... gotowy={gotowy}")
    return gotowy


def przetworz_plik():
    print("Plik gotowy -- przetwarzam dane.")


with DAG(
    dag_id="m3_sensor_demo",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    czekaj_na_plik = PythonSensor(
        task_id="czekaj_na_plik",
        python_callable=plik_gotowy,
        poke_interval=2,   # co ile sekund sprawdzac (w produkcji czesto 60+)
        timeout=15,        # po ilu sekundach poddac sie z bledem Timeout
        mode="poke",       # "poke" blokuje slot workera; "reschedule" go zwalnia miedzy probami
    )
    t_przetworz = PythonOperator(task_id="przetworz_plik", python_callable=przetworz_plik)

    czekaj_na_plik >> t_przetworz
'''

with open(f"{AIRFLOW_HOME}/dags/m3_sensor_demo.py", "w") as f:
    f.write(dag_kod)

# tworzymy plik OD RAZU, zeby sensor "zlapal" go przy pierwszym sprawdzeniu w tescie
with open("dane_wejsciowe.csv", "w") as f:
    f.write("a,b\n1,2\n")

airflow("dags", "test", "m3_sensor_demo", "2024-01-01")


> ⚠️ **`poke` kontra `reschedule`**
>
> Tryb `poke` (domyślny) trzyma slot workera ZAJĘTY przez cały czas oczekiwania -- dobre dla krótkich oczekiwań (sekundy/minuty). Tryb `mode="reschedule"` zwalnia slot między próbami i "budzi się" tylko na chwilę sprawdzenia -- lepsze, gdy sensor może czekać GODZINAMI (np. na plik, który spłynie dopiero wieczorem), żeby nie blokować zasobów klastra Airflow na cały ten czas.

<a id="sec2"></a>
## 2. Branching — wybór ścieżki w zależności od danych

`BranchPythonOperator` uruchamia funkcję, która ZWRACA `task_id` (albo listę `task_id`)
zadania/zadań, które mają się wykonać DALEJ -- wszystkie pozostałe ścieżki zostają
automatycznie oznaczone jako `skipped` (pominięte), a nie `failed`.

In [ ]:
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from datetime import datetime


def policz_liczbe_zamowien():
    return 50  # w prawdziwym DAG-u odczytane z bazy danych


def wybierz_sciezke(ti):
    liczba = ti.xcom_pull(task_ids="policz_liczbe_zamowien")
    if liczba > 100:
        return "sciezka_duzy_wolumen"
    return "sciezka_maly_wolumen"


def przetwarzanie_rownolegle():
    print("Duzy wolumen -- uzywam Sparka do przetwarzania rownoleglego.")


def przetwarzanie_proste():
    print("Maly wolumen -- wystarczy prosty skrypt Pythona.")


with DAG(
    dag_id="m3_branching_demo",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_policz = PythonOperator(task_id="policz_liczbe_zamowien", python_callable=policz_liczbe_zamowien)
    t_branch = BranchPythonOperator(task_id="wybierz_sciezke", python_callable=wybierz_sciezke)
    t_duzy = PythonOperator(task_id="sciezka_duzy_wolumen", python_callable=przetwarzanie_rownolegle)
    t_maly = PythonOperator(task_id="sciezka_maly_wolumen", python_callable=przetwarzanie_proste)

    t_policz >> t_branch >> [t_duzy, t_maly]
'''

with open(f"{AIRFLOW_HOME}/dags/m3_branching_demo.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m3_branching_demo", "2024-01-01")


> ⚡ **50 zamówień -> `sciezka_maly_wolumen`**
>
> W powyższym uruchomieniu funkcja zwraca `50`, więc branching wybiera `sciezka_maly_wolumen` -- w logach zobaczysz komunikat "Maly wolumen..." ORAZ informację, że `sciezka_duzy_wolumen` została pominięta (`skipped`), a nie uruchomiona.

<a id="sec3"></a>
## 3. Dynamic Task Mapping

Czasem liczba zadań NIE jest znana z góry -- np. chcesz przetworzyć każdy plik z listy,
której długość zależy od danych. **Dynamic Task Mapping** (`.partial()` +
`.expand()`, dostępne od Airflow 2.3) tworzy TYLE kopii zadania, ile elementów ma
przekazana lista -- każda kopia dostaje swój własny element i działa niezależnie.

In [ ]:
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def wygeneruj_liste_miast():
    return ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]


def przetworz_miasto(miasto):
    print(f"Przetwarzam dane dla miasta: {miasto}")


with DAG(
    dag_id="m3_dynamic_mapping_demo",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_lista = PythonOperator(task_id="wygeneruj_liste_miast", python_callable=wygeneruj_liste_miast)

    # .partial() ustawia argumenty WSPOLNE dla wszystkich kopii,
    # .expand() tworzy PO JEDNEJ kopii zadania na kazdy element listy
    t_mapowane = PythonOperator.partial(
        task_id="przetworz_miasto",
        python_callable=przetworz_miasto,
    ).expand(op_args=t_lista.output.map(lambda miasto: [miasto]))
'''

with open(f"{AIRFLOW_HOME}/dags/m3_dynamic_mapping_demo.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m3_dynamic_mapping_demo", "2024-01-01")


> 🏭 **Dynamic mapping w prawdziwym potoku**
>
> Typowe zastosowanie: masz listę plików/tabel/klientów o zmiennej długości (np. "wszystkie pliki, które wylądowały dzisiaj w magazynie") i chcesz przetworzyć KAŻDY osobnym zadaniem (z osobnym statusem, osobnymi logami, osobnym retry) -- bez dynamic mapping musiałbyś z góry zgadywać maksymalną liczbę zadań i tworzyć je ręcznie w pętli Pythona przy budowaniu DAG-a.

<a id="sec4"></a>
## 4. Retries i trigger rules

**Retries.** Każdy task może automatycznie ponowić próbę po błędzie -- parametry
`retries` (ile razy) i `retry_delay` (jak długo czekać między próbami) ustawia się per
task albo globalnie przez `default_args` DAG-a.

In [ ]:
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta


def czasem_zawodzi():
    print("Zadanie probuje sie wykonac (w tym przykladzie od razu sie udaje).")


with DAG(
    dag_id="m3_retry_demo",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
    default_args={
        "retries": 3,
        "retry_delay": timedelta(minutes=5),
    },
) as dag:
    t = PythonOperator(task_id="czasem_zawodzi", python_callable=czasem_zawodzi)
'''

with open(f"{AIRFLOW_HOME}/dags/m3_retry_demo.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m3_retry_demo", "2024-01-01")


**Trigger rules** decydują, KIEDY task ma się uruchomić w zależności od statusu
zadań, na których zależy (domyślnie: `all_success` -- wszystkie poprzedniki muszą się
udać). Najczęściej używane alternatywy:

- `all_done` -- uruchom niezależnie od tego, czy poprzedniki się udały czy zawiodły
  (typowe dla kroku "posprzątaj/wyślij powiadomienie o zakończeniu").
- `one_failed` -- uruchom, jeśli PRZYNAJMNIEJ JEDEN poprzednik zawiódł (typowe dla
  ścieżki "wyślij alert o błędzie").
- `all_failed` -- uruchom tylko, jeśli WSZYSTKIE poprzedniki zawiodły.

In [ ]:
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def krok_1():
    print("Krok 1 wykonany.")


def krok_2():
    print("Krok 2 wykonany.")


def posprzataj():
    print("Sprzatanie -- uruchamia sie NIEZALEZNIE od statusu krokow 1 i 2.")


with DAG(
    dag_id="m3_trigger_rule_demo",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t1 = PythonOperator(task_id="krok_1", python_callable=krok_1)
    t2 = PythonOperator(task_id="krok_2", python_callable=krok_2)
    t_sprzatanie = PythonOperator(
        task_id="posprzataj",
        python_callable=posprzataj,
        trigger_rule="all_done",
    )

    [t1, t2] >> t_sprzatanie
'''

with open(f"{AIRFLOW_HOME}/dags/m3_trigger_rule_demo.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m3_trigger_rule_demo", "2024-01-01")


<a id="sec5"></a>
## 5. Wywoływanie narzędzi zewnętrznych z Airflow

W praktyce Airflow rzadko robi "prawdziwą pracę" sam w Pythonie -- zwykle jego zadaniem
jest wywołać ZEWNĘTRZNE narzędzie (dbt, `spark-submit`, skrypt Pythona, zapytanie do API)
i sprawdzić, czy się udało. `BashOperator` jest tu najbardziej uniwersalnym mostem. Poniżej
symulujemy wywołanie zewnętrznego skryptu przetwarzającego dane -- w Module 4 zastąpimy
taki skrypt PRAWDZIWYM wywołaniem `dbt run`.

In [ ]:
# Najpierw tworzymy prosty, "zewnetrzny" skrypt Pythona -- symulacje np. joba Sparka
skrypt = '''
import sys
print(f"Zewnetrzny skrypt przetwarza dane dla segmentu: {sys.argv[1]}")
print("Przetwarzanie zakonczone sukcesem.")
'''
with open("przetworz_segment.py", "w") as f:
    f.write(skrypt)

dag_kod = '''
from airflow import DAG
from airflow.operators.bash import BashOperator
from datetime import datetime
import os

SKRYPT = os.path.abspath("przetworz_segment.py")

with DAG(
    dag_id="m3_wywolanie_zewnetrzne",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_premium = BashOperator(
        task_id="przetworz_premium",
        bash_command=f"python3 {SKRYPT} Premium",
    )
    t_standard = BashOperator(
        task_id="przetworz_standard",
        bash_command=f"python3 {SKRYPT} Standardowy",
    )

    [t_premium, t_standard]
'''

with open(f"{AIRFLOW_HOME}/dags/m3_wywolanie_zewnetrzne.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m3_wywolanie_zewnetrzne", "2024-01-01")


> 🏭 **Ten sam wzorzec dla dbt i Sparka**
>
> `BashOperator(bash_command="dbt run --select nazwa_modelu")` albo `BashOperator(bash_command="spark-submit potok.py")` wyglądają DOKŁADNIE tak samo jak powyżej -- Airflow po prostu uruchamia komendę i sprawdza kod wyjścia (0 = sukces). W Module 4 zobaczymy to naprawdę, wywołując `dbt run` z poziomu DAG-a.

<a id="sec6"></a>
## 6. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- sensory (`PythonSensor`) i różnicę między trybem `poke` a `reschedule`,
- branching (`BranchPythonOperator`) -- wybór ścieżki na podstawie danych, z automatycznym
  pomijaniem (`skipped`) niewybranych gałęzi,
- dynamic task mapping (`.partial()` + `.expand()`) -- tworzenie zadań dynamicznie, na
  podstawie długości listy nieznanej z góry,
- retries i trigger rules (`all_done`, `one_failed`, `all_failed`),
- wywoływanie narzędzi zewnętrznych (`BashOperator`) -- most między Airflow a dbt/Sparkiem.

### 📝 Ćwiczenie 1: Sensor czekający na dwa pliki

Napisz DAG `m3_cw1_dwa_pliki` z `PythonSensor`, który kończy się sukcesem dopiero, gdy ISTNIEJĄ OBA pliki: `plik_a.csv` ORAZ `plik_b.csv` (utwórz je w tej samej komórce PRZED uruchomieniem testu, żeby sensor zadziałał od razu).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
with open("plik_a.csv", "w") as f:
    f.write("a\n1\n")
with open("plik_b.csv", "w") as f:
    f.write("b\n2\n")

dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.sensors.python import PythonSensor
from datetime import datetime
import os


def oba_pliki_gotowe():
    return os.path.exists("plik_a.csv") and os.path.exists("plik_b.csv")


def przetworz():
    print("Oba pliki gotowe -- przetwarzam.")


with DAG(
    dag_id="m3_cw1_dwa_pliki",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    czekaj = PythonSensor(task_id="czekaj_na_oba", python_callable=oba_pliki_gotowe, poke_interval=2, timeout=15)
    t = PythonOperator(task_id="przetworz", python_callable=przetworz)
    czekaj >> t
'''
with open(f"{AIRFLOW_HOME}/dags/m3_cw1_dwa_pliki.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m3_cw1_dwa_pliki", "2024-01-01")
```

</details>

### 📝 Ćwiczenie 2: Branching na trzy ścieżki

Rozbuduj wzorzec z sekcji 2 do TRZECH ścieżek zależnie od segmentu klienta (`"Nowy"`, `"Standardowy"`, `"Premium"`) -- funkcja branchingu ma zwracać odpowiedni `task_id` w zależności od zmiennej `segment = "Premium"`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from datetime import datetime

segment = "Premium"


def wybierz_segment():
    mapa = {"Nowy": "sciezka_nowy", "Standardowy": "sciezka_standardowy", "Premium": "sciezka_premium"}
    return mapa[segment]


def obsluz(nazwa):
    def wykonaj():
        print(f"Obsluga segmentu: {nazwa}")
    return wykonaj


with DAG(
    dag_id="m3_cw2_trzy_sciezki",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    branch = BranchPythonOperator(task_id="wybierz_segment", python_callable=wybierz_segment)
    a = PythonOperator(task_id="sciezka_nowy", python_callable=obsluz("Nowy"))
    b = PythonOperator(task_id="sciezka_standardowy", python_callable=obsluz("Standardowy"))
    c = PythonOperator(task_id="sciezka_premium", python_callable=obsluz("Premium"))
    branch >> [a, b, c]
'''
with open(f"{AIRFLOW_HOME}/dags/m3_cw2_trzy_sciezki.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m3_cw2_trzy_sciezki", "2024-01-01")
```

</details>

### 📝 Ćwiczenie 3: Dynamic mapping nad listą kategorii produktów

Napisz DAG z dynamic task mapping nad listą pięciu kategorii produktów (`["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]`), gdzie każda kopia zadania wypisuje `f"Licze sprzedaz dla kategorii: {kategoria}"`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def lista_kategorii():
    return ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]


def policz_sprzedaz(kategoria):
    print(f"Licze sprzedaz dla kategorii: {kategoria}")


with DAG(
    dag_id="m3_cw3_kategorie",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_lista = PythonOperator(task_id="lista_kategorii", python_callable=lista_kategorii)
    t_mapowane = PythonOperator.partial(
        task_id="policz_sprzedaz",
        python_callable=policz_sprzedaz,
    ).expand(op_args=t_lista.output.map(lambda k: [k]))
'''
with open(f"{AIRFLOW_HOME}/dags/m3_cw3_kategorie.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m3_cw3_kategorie", "2024-01-01")
```

</details>

> 🔥 **Wyzwanie: potok z sensorem, branchingiem i sprzątaniem**
>
> Zbuduj DAG `m3_wyzwanie_potok`, który: (1) CZEKA sensorem na plik `dane_gotowe.csv` (utwórz go w tej samej komórce), (2) wybiera branchingiem ścieżkę w zależności od tego, czy liczba wierszy w pliku jest większa niż 2 (użyj `"duzy_plik"`/`"maly_plik"` jako task_id), (3) niezależnie od wybranej ścieżki, na końcu uruchamia task `"zakoncz"` z `trigger_rule="all_done"`, który wypisuje "Potok zakonczony".

<details>
<summary>Pokaż rozwiązanie</summary>

```python
with open("dane_gotowe.csv", "w") as f:
    f.write("a,b\n1,2\n3,4\n5,6\n")

dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.sensors.python import PythonSensor
from datetime import datetime
import os


def plik_gotowy():
    return os.path.exists("dane_gotowe.csv")


def wybierz_sciezke():
    with open("dane_gotowe.csv") as f:
        liczba_wierszy = sum(1 for _ in f) - 1  # -1 za naglowek
    return "duzy_plik" if liczba_wierszy > 2 else "maly_plik"


def duzy_plik():
    print("Duzy plik -- przetwarzanie rownolegle.")


def maly_plik():
    print("Maly plik -- przetwarzanie proste.")


def zakoncz():
    print("Potok zakonczony")


with DAG(
    dag_id="m3_wyzwanie_potok",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    czekaj = PythonSensor(task_id="czekaj_na_plik", python_callable=plik_gotowy, poke_interval=2, timeout=15)
    branch = BranchPythonOperator(task_id="wybierz_sciezke", python_callable=wybierz_sciezke)
    duzy = PythonOperator(task_id="duzy_plik", python_callable=duzy_plik)
    maly = PythonOperator(task_id="maly_plik", python_callable=maly_plik)
    koniec = PythonOperator(task_id="zakoncz", python_callable=zakoncz, trigger_rule="all_done")

    czekaj >> branch >> [duzy, maly] >> koniec
'''
with open(f"{AIRFLOW_HOME}/dags/m3_wyzwanie_potok.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m3_wyzwanie_potok", "2024-01-01")
```

trigger_rule="all_done" na tasku koncowym to typowy wzorzec "sprzataj/podsumuj niezaleznie od wyniku" -- bez niego task 'zakoncz' czekalby na SUKCES obu galezi branchingu, a jedna z nich zawsze jest pomijana (skipped), wiec domyslna regula all_success by go zablokowala.

</details>

## Co dalej?

W **Module 4** poznajemy **dbt** -- narzędzie do transformacji danych czystym SQL-em
wewnątrz hurtowni. Zaczniemy budować modele, które w Module 10 (projekt końcowy) Airflow
będzie orkiestrował dokładnie tak, jak w sekcji 5 tego modułu.